# STIR-Net V1 — 13 real historical-evidence diagnostic

This notebook tests the new historical-instance evidence pathway on the **same real BlastoSPIM scene used in Notebook 12**, without rerunning Notebook 12.

It does four things:

1. builds **genuine** `4×12×12×12` historical instance grids from `raw_movie`, `instance_movie`, and the Trackastra graph;
2. rebuilds the temporal graph so the new 22-D hypothesis evidence, projected past/future supports, and component-overlap evidence are populated;
3. migrates the **Notebook-12 spatial-dense checkpoint** into the new history-aware architecture, then continues the same curriculum from `temporal_dense` onward;
4. compares the trained full-history model against Notebook-12 saved diagnostics and performs same-weights destructive perturbations (`correct`, `zero`, `shuffled`, etc.) to prove whether the model actually uses history.

## Decisive questions

- Is real historical evidence present for the giant merged source 9?
- Do the history encoder, fusion gate, 22-D dynamics, and support-attention bias receive gradients?
- Does source-9 sibling collapse improve compared with Notebook 12?
- Do split-query centers become more distinct?
- Does Hungarian assignment remain stable or churn between GT cells?
- With the **same trained weights**, does correct history outperform zero/shuffled history?

A positive answer to the last question is the strongest evidence that the new architecture is genuinely useful.

In [ ]:
from pathlib import Path
from collections import defaultdict
import gc, json, pickle, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from scipy import ndimage as ndi
from scipy.stats import rankdata

from learned.stirnet import RefinementCriterion, StirNet
from learned.stirnet.data.graph_builder import AssociationRecord, DetectionRecord, build_temporal_graph
from learned.stirnet.data.historical_instances import build_historical_instance_grid
from learned.stirnet.data.sample_builder import robust_normalize
from learned.stirnet.data.targets import extract_instance_metadata
from learned.stirnet.debugging.acceptance.first_overfit import (
    _reduced_config, _repo_root, _roi_with_all_cells, build_real_batch,
)
from learned.stirnet.model.query_builder import QUERY_PRIMARY, QUERY_SPLIT, QUERY_TEMPORAL, QUERY_DISCOVERY
from learned.stirnet.training.checkpoint import load_checkpoint, save_checkpoint
from learned.stirnet.training.trainer import Trainer, model_forward_from_batch, move_batch_to_device

SEED=40266
QUERY_NAMES={QUERY_PRIMARY:'primary',QUERY_SPLIT:'split',QUERY_TEMPORAL:'temporal',QUERY_DISCOVERY:'discovery'}

# Keep the post-spatial schedule identical to Notebook 12 for a meaningful comparison.
SPATIAL_DENSE_STEPS=30
TEMPORAL_DENSE_STEPS=20
QUERY_BOOTSTRAP_STEPS=20
NATIVE_BOOTSTRAP_STEPS=10
JOINT_STEPS=5
LOG_EVERY=2
ASSIGNMENT_EVERY=2

REPO_ROOT=_repo_root(Path.cwd())
DATA_DIR=REPO_ROOT/'data'/'learned'/'stirnet'/'first_overfit'/'BlastoSPIM1_F22_030_034'
NB12_DIR=REPO_ROOT/'runs'/'stirnet'/'first_overfit'/'12_staged_same_sample'
NB12_SPATIAL_CKPT=NB12_DIR/'checkpoint_spatial_dense.pt'
RUN_DIR=REPO_ROOT/'runs'/'stirnet'/'debugging'/'13_real_history_diagnostic'
RUN_DIR.mkdir(parents=True,exist_ok=True)

HISTORY_CACHE_DIR = DATA_DIR / "history_v2"
HISTORY_CACHE_DIR.mkdir(parents=True, exist_ok=True)

TEMPORAL_CACHE = (
    HISTORY_CACHE_DIR
    / "real_history_temporal_v2.pt"
)

np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
if not torch.cuda.is_available(): raise RuntimeError('Notebook 13 requires CUDA.')
device=torch.device('cuda')

print('Repo:',REPO_ROOT)
print('GPU:',torch.cuda.get_device_name(0))
print('Notebook-12 spatial checkpoint:',NB12_SPATIAL_CKPT, NB12_SPATIAL_CKPT.exists())
if not NB12_SPATIAL_CKPT.exists():
    raise FileNotFoundError('Notebook 12 spatial-dense checkpoint is required. It should exist from the completed Notebook-12 run.')


## 1. Load the same current-frame scene

`build_real_batch()` is used only for the current-frame spatial inputs, current CC labels, instance metadata, and GT target. We replace its temporal tensors with a history-aware reconstruction below.

In [ ]:
base_batch,sample=build_real_batch(DATA_DIR)
assert sample['current_count']==36,sample
assert sample['target_count']==33,sample
assert sample['temporal_tracklets']==52,sample
print(json.dumps(sample,indent=2,default=float))


## 2. Build genuine historical grids and the new temporal graph

This is notebook-local for now. Each Trackastra detection gets a compact history grid centered at the Trackastra position in the common ROI. `build_temporal_graph()` then constructs nearest past/future support, projected component overlap, and the 22-D hypothesis graph.

In [ ]:
def build_real_history_temporal(data_dir,cfg,use_cache=True):
    if use_cache and TEMPORAL_CACHE.exists():
        print('loading cache',TEMPORAL_CACHE)
        return torch.load(TEMPORAL_CACHE,map_location='cpu',weights_only=False)

    data_dir=Path(data_dir); src=data_dir/'stirnet_source'; trdir=data_dir/'trackastra'
    raw_movie=np.load(data_dir/'raw_movie.npy',mmap_mode='r')
    instance_movie=np.load(data_dir/'instance_movie.npy',mmap_mode='r')
    markers_movie=np.load(data_dir/'markers_movie.npy',mmap_mode='r')
    gt_movie=np.load(data_dir/'gt_movie.npy',mmap_mode='r')
    with open(data_dir/'metadata.json','r',encoding='utf-8') as f: meta=json.load(f)
    with open(trdir/'track_graph.pkl','rb') as f: track_graph=pickle.load(f)
    spacing=np.asarray(meta['spacing_zyx_um'],np.float32);dref=float(np.load(src/'dref_um.npy'));target_t=2
    roi,roi_low,roi_high=_roi_with_all_cells(instance_movie,gt_movie,spacing)
    roi_shape=(roi_high-roi_low).astype(np.int64);roi_center=.5*(roi_shape.astype(np.float32)-1)*spacing
    full_shape=np.asarray(instance_movie.shape[-3:],np.float32)
    current=np.asarray(instance_movie[target_t][roi]).astype(np.int32,copy=True)
    node_pos={int(n):np.asarray(d['coords'],np.float32)*spacing for n,d in track_graph.nodes(data=True)}

    def velocity(node,neigh,forward):
        if not neigh:return np.zeros(3,np.float32)
        t0=int(track_graph.nodes[node]['time']);p0=node_pos[node];vals=[]
        for o in neigh:
            o=int(o);dt=abs(int(track_graph.nodes[o]['time'])-t0)
            if dt:
                delta=node_pos[o]-p0;vals.append((delta if forward else -delta)/dt)
        return np.mean(vals,axis=0).astype(np.float32) if vals else np.zeros(3,np.float32)

    records=[];build_rows=[]
    for t in range(len(instance_movie)):
        labels=np.asarray(instance_movie[t][roi]).astype(np.int32,copy=False)
        raw=robust_normalize(np.asarray(raw_movie[t][roi]));marker=(np.asarray(markers_movie[t][roi])>0).astype(np.float32)
        imeta=extract_instance_metadata(labels,raw,tuple(spacing),dref,marker)
        ids=imeta.ids.numpy();features=imeta.features.numpy();id2row={int(v):i for i,v in enumerate(ids)}
        nodes=[(int(n),d) for n,d in track_graph.nodes(data=True) if int(d['time'])==t]
        print(f'frame {t}: {len(nodes)} Trackastra nodes')
        for node,nd in nodes:
            lid=int(nd['label'])
            if lid not in id2row: continue
            row=id2row[lid];coord_full=np.asarray(nd['coords'],np.float32);coord_roi=coord_full-roi_low.astype(np.float32)
            center_roi_um=coord_roi*spacing;pos_rel=center_roi_um-roi_center
            grid,ok=build_historical_instance_grid(raw,labels,lid,tuple(spacing),dref,center_um=center_roi_um,
                                                    grid_size=cfg.history.grid_size,extent_dref=cfg.history.extent_dref)
            grid=grid.to(torch.float16);vox=int(np.count_nonzero(labels==lid));feat=features[row]
            lf=coord_full*spacing;uf=(full_shape-1-coord_full)*spacing;lr=coord_roi*spacing;ur=(roi_shape.astype(np.float32)-1-coord_roi)*spacing
            pred=list(track_graph.predecessors(node));succ=list(track_graph.successors(node));dist=float(np.min(np.concatenate([lf,uf])))
            records.append(DetectionRecord(
                node_id=node,time_offset=t-target_t,position_um=tuple(pos_rel.tolist()),physical_volume_um3=vox*float(np.prod(spacing)),
                bbox_um=tuple((feat[1:4]*dref).tolist()),pca_axes_um=tuple((feat[4:7]*dref).tolist()),
                elongation=float(feat[7]),flatness=float(feat[8]),solidity=float(feat[9]),compactness=float(feat[10]),
                intensity_mean=float(feat[11]),intensity_std=float(feat[12]),
                backward_velocity_um=tuple(velocity(node,pred,False).tolist()),forward_velocity_um=tuple(velocity(node,succ,True).tolist()),
                distance_to_volume_boundary_um=dist,distance_to_patch_boundary_um=float(np.min(np.concatenate([lr,ur]))),
                boundary_related=dist<=4.0,instance_grid=grid,history_valid=bool(ok)))
            build_rows.append({'node_id':node,'time_offset':t-target_t,'label_id':lid,'valid':bool(ok),
                               'grid_abs_mean':float(grid.float().abs().mean()),'nonzero_frac':float((grid!=0).float().mean())})
        del labels,raw,marker,imeta;gc.collect()

    associations=[]
    for s,d,ed in track_graph.edges(data=True):
        s=int(s);d=int(d);score=ed.get('weight')
        associations.append(AssociationRecord(s,d,None if score is None else float(score),
                         'division' if track_graph.out_degree(s)>1 else 'temporal'))

    temporal=build_temporal_graph(records,associations,dref_um=dref,temporal_radius=cfg.temporal.temporal_radius,
             k_spatial_neighbors=cfg.temporal.k_spatial_neighbors,spatial_radius_dref=cfg.temporal.spatial_neighbor_radius_dref,
             current_labels=current,spacing_um=tuple(spacing),history_extent_dref=cfg.history.extent_dref)
    temporal['_build_rows']=build_rows
    torch.save(temporal,TEMPORAL_CACHE);print('saved',TEMPORAL_CACHE)
    return temporal

cfg=_reduced_config();cfg.history.enabled=True
hist_temporal=build_real_history_temporal(DATA_DIR,cfg,use_cache=True)
build_rows=pd.DataFrame(hist_temporal.pop('_build_rows',[]))
batch=dict(base_batch);batch.update(hist_temporal);batch['temporal_batch']=torch.zeros(len(batch['temporal_ref_um']),dtype=torch.long)
print('history-aware temporal tracklets:',len(batch['temporal_ref_um']))


## 3. Hard preflight — history must actually exist

If this fails, stop. A history-on model with zero/invalid tensors would be a meaningless experiment.

In [ ]:
nv=batch['node_history_valid'].bool();sv=batch['history_support_valid'].bool();ea=batch['hypothesis_edge_attr']
preflight={
 'graph_nodes':len(batch['graph_x']),'valid_node_history':int(nv.sum()),'node_history_fraction':float(nv.float().mean()),
 'temporal_tracklets':len(batch['temporal_ref_um']),'past_support_valid':int(sv[:,0].sum()),'future_support_valid':int(sv[:,1].sum()),
 'support_entries_valid':int(sv.sum()),'nonzero_node_grid_values':int(torch.count_nonzero(batch['node_instance_grid'][nv])),
 'nonzero_support_values':int(torch.count_nonzero(batch['history_support'][sv])),'hypothesis_edge_dim':int(ea.shape[-1]),
 'nonzero_new_edge_features':int(torch.count_nonzero(ea[:,8:22])),'component_assigned':int((batch['best_current_component_id']>0).sum()),
 'positive_component_overlap':int((batch['best_component_overlap']>0).sum()),
}
display(pd.DataFrame([preflight]).T)
assert preflight['valid_node_history']>0
assert preflight['support_entries_valid']>0
assert preflight['nonzero_node_grid_values']>0
assert preflight['nonzero_support_values']>0
assert preflight['hypothesis_edge_dim']==22
print('REAL HISTORY PREFLIGHT PASSED')
if len(build_rows):display(build_rows.groupby('time_offset').agg(records=('node_id','count'),valid=('valid','sum'),mean_abs=('grid_abs_mean','mean'),nonzero=('nonzero_frac','mean')).reset_index())


## 4. Does the temporal evidence actually point into source 9?

This is an upstream-information check. If only one tracklet projects onto the nine-cell merge, the neural pathway cannot magically recover nine identities from history.

In [ ]:
def component_tracklet_table(b,sid=9):
    comp=b['best_current_component_id'].long();idx=torch.nonzero(comp==sid,as_tuple=False).flatten();rows=[]
    refs=b['temporal_ref_um'];best=b['best_component_overlap'];second=b['second_best_component_overlap'];sv=b['history_support_valid'];dt=b['history_support_dt']
    ei=b['hypothesis_edge_index'];ea=b['hypothesis_edge_attr']
    for m in idx.tolist():
        inc=(ei[0]==m)|(ei[1]==m);a=ea[inc]
        rows.append({'tracklet':m,'ref_z_um':float(refs[m,0]),'ref_y_um':float(refs[m,1]),'ref_x_um':float(refs[m,2]),
          'best_overlap':float(best[m]),'second_overlap':float(second[m]),'past_valid':bool(sv[m,0]),'future_valid':bool(sv[m,1]),
          'past_dt':float(dt[m,0]),'future_dt':float(dt[m,1]),'incident_edges':int(inc.sum()),
          'mean_same_component_conf':float(a[:,4].mean()) if len(a) else np.nan,
          'mean_closing_speed':float(a[:,11].mean()) if len(a) else np.nan,
          'mean_projected_support_overlap':float(a[:,15].mean()) if len(a) else np.nan})
    return pd.DataFrame(rows)

s9_history=component_tracklet_table(batch,9)
print('tracklets projected to source 9:',len(s9_history));display(s9_history)
s9_history.to_csv(RUN_DIR/'source9_history_evidence.csv',index=False)
if len(s9_history)<2: print('WARNING: very little historical identity evidence reaches source 9.')


## 5. Configure the production curriculum and migrate the Notebook-12 spatial checkpoint

The spatial stage is **not rerun**. New history modules receive their conservative initialization via checkpoint migration; the legacy hypothesis-edge projection keeps columns 0–7 and initializes new columns 8–21 to zero.

In [ ]:
cfg=_reduced_config();cfg.history.enabled=True
cfg.curriculum.enabled=True;cfg.curriculum.spatial_dense_steps=SPATIAL_DENSE_STEPS;cfg.curriculum.temporal_dense_steps=TEMPORAL_DENSE_STEPS
cfg.curriculum.query_bootstrap_steps=QUERY_BOOTSTRAP_STEPS;cfg.curriculum.native_bootstrap_steps=NATIVE_BOOTSTRAP_STEPS

torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
model=StirNet(cfg).to(device)
ckpt=load_checkpoint(NB12_SPATIAL_CKPT,model,map_location=device,strict=True,migrate_history=True)
print('checkpoint step:',ckpt.get('step'))
print('history migration notes:',len(ckpt.get('history_migration',[])))
for line in ckpt.get('history_migration',[])[:12]:print(' ',line)

trainer=Trainer(model,cfg,device=device,amp_dtype='fp16')
trainer.global_step=SPATIAL_DENSE_STEPS;trainer.curriculum.current=None
b=move_batch_to_device(batch,device)
print('starting curriculum stage:',trainer.curriculum.apply(trainer.global_step).name)


## 6. Diagnostic helpers — matching, history usage, and source-9 collapse

In [ ]:
target=batch['targets'][0];target_ids=torch.as_tensor(target['ids']).cpu().long();target_centers=torch.as_tensor(target['centers_cellscale']).float().cpu()
source_ids_meta=torch.as_tensor(target['source_ids']).cpu().long();source_overlap=torch.as_tensor(target['source_gt_overlap']).cpu();source_row={int(v):i for i,v in enumerate(source_ids_meta.tolist())}
gt=np.asarray(torch.as_tensor(target['label_map']).cpu(),dtype=np.int32);gt_flat=gt.reshape(-1);current=np.asarray(batch['instance_labels'][0],dtype=np.int32);current_flat=current.reshape(-1)
spacing=np.asarray(batch['spacing_um'][0],np.float32);dref=float(batch['dref_um'][0]);extent=(np.asarray(current.shape,np.float32)-1)*spacing

def output_dict(o):
    d={n:getattr(o,n) for n in getattr(o,'__dataclass_fields__',{}) if torch.is_tensor(getattr(o,n))}
    d.update(exist_logits=o.exist_logits,centers_cellscale=o.centers_cellscale,coarse_mask_logits=o.coarse_mask_logits,
             coarse_spacing_um=o.coarse_spacing_um,dref_um=o.dref_um,query_types=o.query_types,source_instance_ids=o.source_instance_ids,
             query_initial_references_cellscale=o.query_initial_references_cellscale)
    return d

def get_matches(o,criterion):
    final=output_dict(o);ct=criterion._coarse_targets(final,b['targets']);return criterion._match(final,o.query_padding_mask,b['targets'],ct)

def roc_auc(y,s):
    y=np.asarray(y);s=np.asarray(s);p=y==1;n=y==0
    if not p.any() or not n.any():return np.nan
    r=rankdata(s,method='average');return float((r[p].sum()-p.sum()*(p.sum()+1)/2)/(p.sum()*n.sum()))

def pair_cos(x):
    if len(x)<2:return np.nan
    x=F.normalize(x.float(),dim=-1);m=x@x.T;mask=~torch.eye(len(x),dtype=torch.bool,device=x.device);return float(m[mask].mean().detach().cpu())

def center_stats(x):
    if len(x)<2:return dict(center_mean_um=np.nan,center_median_um=np.nan,center_min_um=np.nan)
    d=torch.pdist(x.float())*dref;return dict(center_mean_um=float(d.mean()),center_median_um=float(d.median()),center_min_um=float(d.min()))

def soft_pair_dice(prob):
    if len(prob)<2:return np.nan
    f=prob.float().flatten(1);inter=2*(f[:,None]*f[None,:]).sum(-1);den=f.sum(-1)[:,None]+f.sum(-1)[None,:];d=(inter+1e-6)/(den+1e-6)
    mask=~torch.eye(len(f),dtype=torch.bool,device=f.device);return float(d[mask].mean().detach().cpu())

def seed_queries(o,sid):
    qt=o.query_types[0].detach().cpu();si=o.source_instance_ids[0].detach().cpu();valid=~o.query_padding_mask[0].detach().cpu()
    return [int(q) for q in torch.nonzero(valid,as_tuple=False).flatten().tolist() if int(si[q])==sid and int(qt[q]) in (QUERY_PRIMARY,QUERY_SPLIT)]

def source_gt_dice(sid,t):
    gid=int(target_ids[t]);a=current_flat==sid;c=gt_flat==gid;inter=np.count_nonzero(a&c);return 2*inter/max(int(a.sum())+int(c.sum()),1)

def native_query_mask(o,q):
    idx=torch.tensor([q],device=device);logit=model.render_masks(o,[idx])[0][0];p=torch.sigmoid(logit.float()).cpu().numpy();mask=p>cfg.inference.mask_threshold
    center=o.centers_cellscale[0,q].detach().float().cpu().numpy()*dref;cv=(center+.5*extent)/spacing
    cc,n=ndi.label(mask)
    if n:
        c=np.rint(cv).astype(int);lab=int(cc[tuple(np.minimum(np.maximum(c,0),np.asarray(mask.shape)-1))]);
        if lab>0:mask=cc==lab
    return np.flatnonzero(mask.reshape(-1))

def sparse_dice(a,b):
    inter=len(np.intersect1d(a,b,assume_unique=True));return 2*inter/max(len(a)+len(b),1)

def q_gt_dice(idx,t):
    gid=int(target_ids[t]);cnt=int(np.count_nonzero(gt_flat==gid));inter=int(np.count_nonzero(gt_flat[idx]==gid)) if len(idx) else 0
    return 2*inter/max(len(idx)+cnt,1)

def scalar(v):
    if v is None:return np.nan
    if torch.is_tensor(v):return float(v.detach().float().mean().cpu()) if v.numel() else np.nan
    return float(v) if isinstance(v,(int,float,np.number)) else np.nan


## 7. Snapshot evaluator

At every curriculum boundary this captures both task performance and the internal split-collapse trajectory.

In [ ]:
snapshots=[];collapse=[];specialization=[];assignment_history=[]

def forward_debug():
    return model(b['spatial_inputs'],b['instance_labels'],b['spacing_um'],b['dref_um'],b['instance_features'],b['instance_ids'],b['instance_batch'],b['instance_centroids_um'],
      b['graph_x'],b['graph_edge_index'],b['graph_edge_attr'],b['tracklet_id'],b['temporal_ref_um'],b['temporal_status'],b['hypothesis_edge_index'],b['hypothesis_edge_attr'],b['temporal_batch'],b.get('spatial_padding_mask'),
      return_debug=True,bypass_coreasoning=False,node_instance_grid=b.get('node_instance_grid'),node_history_valid=b.get('node_history_valid'),history_support=b.get('history_support'),
      history_support_valid=b.get('history_support_valid'),history_support_dt=b.get('history_support_dt'),history_support_center_um=b.get('history_support_center_um'),history_support_extent_um=b.get('history_support_extent_um'),
      best_current_component_id=b.get('best_current_component_id'),best_component_overlap=b.get('best_component_overlap'),second_best_component_overlap=b.get('second_best_component_overlap'))

@torch.no_grad()
def evaluate(name):
    model.eval();gc.collect();torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats()
    stage=trainer.curriculum.apply(trainer.global_step);trainer.criterion.set_loss_weight_overrides(stage.loss_weight_overrides)
    with torch.autocast('cuda',dtype=torch.float16):o=forward_debug();losses=trainer.criterion(o,b['targets'])
    matches=get_matches(o,trainer.criterion);valid=~o.query_padding_mask[0].detach().cpu();ep=torch.sigmoid(o.exist_logits[0]).detach().float().cpu();mset=set(matches[0].pred_indices.detach().cpu().tolist());vid=torch.nonzero(valid).flatten().tolist()
    y=[int(q in mset) for q in vid];scores=[float(ep[q]) for q in vid];mp=[float(ep[q]) for q in vid if q in mset];up=[float(ep[q]) for q in vid if q not in mset]
    dbg=o.debug or {};hs=dbg.get('history_statistics',{});row={'snapshot':name,'step':trainer.global_step,'stage':stage.name,'loss':float(losses['loss']),
      'coarse_soft_dice':1-float(losses['dice_coarse']),'native_soft_dice':1-float(losses['dice_hi']),
      'exist_matched':float(np.mean(mp)) if mp else np.nan,'exist_unmatched':float(np.mean(up)) if up else np.nan,'exist_auc':roc_auc(y,scores),'accepted_q_0p5':int(((ep>.5)&valid).sum()),
      'matched_count':len(matches[0].pred_indices),'unmatched_gt':sample['target_count']-len(matches[0].pred_indices),'peak_cuda_gib':torch.cuda.max_memory_allocated()/1024**3}
    for k,v in hs.items():row['hist_'+k]=scalar(v)
    gate=dbg.get('history_gate');nv=dbg.get('node_history_valid')
    if gate is not None and nv is not None and nv.bool().any():
        gv=gate[nv.bool()].float();row.update(hist_gate_mean=float(gv.mean()),hist_gate_std=float(gv.std(unbiased=False)),hist_gate_max=float(gv.max()))
    bias=dbg.get('history_support_attention_bias_stats',{})
    for block,stats in bias.items():
        if isinstance(stats,dict):
            for k,v in stats.items():row[f'{block}_bias_{k}']=scalar(v)
    snapshots.append(row);print('\nSNAPSHOT',name);display(pd.DataFrame([row]).T)

    # Source-9 internal representations.
    qs=seed_queries(o,9);qcpu=torch.tensor(qs);initial=o.query_initial_references_cellscale[0].detach().float().cpu()[qcpu]
    collapse.append({'snapshot':name,'representation':'initial_centers',**center_stats(initial)})
    layers=[*o.aux_outputs,{'query_embeddings':o.query_embeddings,'coarse_mask_logits':o.coarse_mask_logits,'centers_cellscale':o.centers_cellscale}]
    for li,lo in enumerate(layers,1):
        emb=lo['query_embeddings'][0,qs];coarse=torch.sigmoid(lo['coarse_mask_logits'][0,qs].float());cent=lo['centers_cellscale'][0,qs].detach().float().cpu()
        collapse.append({'snapshot':name,'representation':f'layer{li}','embedding_cos':pair_cos(emb),'coarse_pair_dice':soft_pair_dice(coarse),**center_stats(cent)})
    masks={q:native_query_mask(o,q) for q in qs};pairs=[]
    for i,q1 in enumerate(qs):
        for q2 in qs[i+1:]:pairs.append(sparse_dice(masks[q1],masks[q2]))
    collapse.append({'snapshot':name,'representation':'native','native_pair_mean':float(np.mean(pairs)),'native_pair_median':float(np.median(pairs))})
    mmap={int(q):int(t) for q,t in zip(matches[0].pred_indices.detach().cpu(),matches[0].target_indices.detach().cpu())};sr=source_row[9];compatible=torch.nonzero(source_overlap[sr]>0).flatten().tolist()
    for q in qs:
        for t in compatible:specialization.append({'snapshot':name,'query':q,'matched_target':mmap.get(q,-1),'gt_target':int(t),'assigned':mmap.get(q,-1)==int(t),'source_dice':source_gt_dice(9,int(t)),'query_dice':q_gt_dice(masks[q],int(t))})
    del o,losses,matches,masks;gc.collect();torch.cuda.empty_cache()


## 8. Baseline after spatial checkpoint, before history training

In [ ]:
evaluate('post_spatial_before_history_training')


## 9. Continue the current production curriculum and record history gradients + assignment churn

In [ ]:
def grad_norm(module):
    if module is None:return np.nan
    vals=[p.grad.detach().float().square().sum() for p in module.parameters() if p.grad is not None]
    return float(torch.stack(vals).sum().sqrt().cpu()) if vals else 0.0

training=[];last_stage=None;total=TEMPORAL_DENSE_STEPS+QUERY_BOOTSTRAP_STEPS+NATIVE_BOOTSTRAP_STEPS+JOINT_STEPS
for local in range(total):
    stage=trainer.curriculum.apply(trainer.global_step);trainer.criterion.set_loss_weight_overrides(stage.loss_weight_overrides)
    if stage.name!=last_stage:
        if last_stage is not None:evaluate('after_'+last_stage)
        last_stage=stage.name;print('\n===',last_stage,'===')
    model.train();trainer.optimizer.zero_grad(set_to_none=True);t0=time.perf_counter()
    with trainer._autocast():
        out=model_forward_from_batch(model,b,bypass_coreasoning=stage.bypass_coreasoning);losses=trainer.criterion(out,b['targets']);loss=losses['loss']
    if (stage.name in {'query_bootstrap','native_bootstrap','joint'} and local%ASSIGNMENT_EVERY==0):
        m=get_matches(out,trainer.criterion);qs=set(seed_queries(out,9))
        for q,t in zip(m[0].pred_indices.detach().cpu().tolist(),m[0].target_indices.detach().cpu().tolist()):
            if q in qs:assignment_history.append({'step':trainer.global_step,'stage':stage.name,'query':q,'target':t})
    trainer.scaler.scale(loss).backward();trainer.scaler.unscale_(trainer.optimizer)
    metrics={k:float(v.detach().float().cpu()) for k,v in losses.items() if torch.is_tensor(v) and v.numel()==1}
    metrics.update(step=trainer.global_step,stage=stage.name,history_encoder_grad=grad_norm(model.history_encoder),history_fusion_grad=grad_norm(model.history_fusion),
                   cr1_history_bias_grad=grad_norm(getattr(model.cr1.cross,'history_bias',None)),cr2_history_bias_grad=grad_norm(getattr(model.cr2.cross,'history_bias',None)))
    gn=torch.nn.utils.clip_grad_norm_(model.parameters(),cfg.training.max_grad_norm);metrics['grad_norm_preclip']=float(gn);trainer.scaler.step(trainer.optimizer);trainer.scaler.update();trainer.global_step+=1;metrics['seconds']=time.perf_counter()-t0;training.append(metrics)
    if local==0 or (local+1)%LOG_EVERY==0 or local+1==total:
        print(stage.name,trainer.global_step,{k:round(metrics[k],6) for k in ['loss','exist','dice_coarse','dice_hi','center','history_encoder_grad','history_fusion_grad','cr1_history_bias_grad'] if k in metrics})
    del out,losses;gc.collect();torch.cuda.empty_cache()
if last_stage is not None:evaluate('after_'+last_stage)

save_checkpoint(RUN_DIR/'history_full_final.pt',model=model,optimizer=trainer.optimizer,scaler=trainer.scaler,step=trainer.global_step,epoch=0,config=cfg,extra={'notebook':13})
training_df=pd.DataFrame(training);snapshot_df=pd.DataFrame(snapshots);collapse_df=pd.DataFrame(collapse);spec_df=pd.DataFrame(specialization);assign_df=pd.DataFrame(assignment_history)
training_df.to_csv(RUN_DIR/'training.csv',index=False);snapshot_df.to_csv(RUN_DIR/'snapshots.csv',index=False);collapse_df.to_csv(RUN_DIR/'source9_collapse.csv',index=False);spec_df.to_csv(RUN_DIR/'source9_specialization.csv',index=False);assign_df.to_csv(RUN_DIR/'source9_assignments.csv',index=False)


## 10. Compare directly against saved Notebook-12 baseline diagnostics

In [ ]:
nb12_snap_path=NB12_DIR/'stage_snapshots.csv';nb12_pair_path=NB12_DIR/'merged_source_pair_overlap.csv';nb12_spec_path=NB12_DIR/'merged_source_specialization.csv'
if nb12_snap_path.exists():
    old=pd.read_csv(nb12_snap_path);old_final=old.iloc[-1].to_dict();new_final=snapshot_df.iloc[-1].to_dict()
    rows=[]
    for metric in ['coarse_soft_dice','native_soft_dice','exist_auc']:
        old_key={'exist_auc':'exist_auc'}.get(metric,metric);rows.append({'metric':metric,'Notebook12_no_history':old_final.get(old_key,np.nan),'Notebook13_full_history':new_final.get(metric,np.nan)})
    display(pd.DataFrame(rows))
else: print('Notebook-12 stage_snapshots.csv not found; skip saved-baseline comparison.')

# Source-9 final native sibling overlap from Notebook 12 vs Notebook 13.
if nb12_pair_path.exists():
    oldp=pd.read_csv(nb12_pair_path);old_s9=oldp[(oldp.snapshot=='after_joint')&(oldp.source_id==9)]
    new_native=collapse_df[(collapse_df.snapshot=='after_joint')&(collapse_df.representation=='native')]
    print('Notebook12 source9 pair Dice mean:',float(old_s9.pair_dice.mean()) if len(old_s9) else np.nan)
    print('Notebook13 source9 pair Dice mean:',float(new_native.native_pair_mean.iloc[-1]) if len(new_native) else np.nan)


## 11. Assignment switching audit

If a split query repeatedly changes GT identity, its supervision is unstable. This tests the Hungarian-churn hypothesis directly.

In [ ]:
if len(assign_df):
    rows=[]
    for q,g in assign_df.sort_values('step').groupby('query'):
        seq=g.target.tolist();rows.append({'query':int(q),'observations':len(seq),'unique_targets':len(set(seq)),'switches':sum(a!=b for a,b in zip(seq[:-1],seq[1:])),'sequence':' -> '.join(map(str,seq))})
    switches=pd.DataFrame(rows);display(switches);switches.to_csv(RUN_DIR/'source9_assignment_switches.csv',index=False)
else: print('No assignment observations.')


## 12. History gradient audit

Nonzero history gradients prove trainability. They do **not** by themselves prove usefulness; the same-weights perturbation test below is stronger.

In [ ]:
display(training_df.groupby('stage')[['history_encoder_grad','history_fusion_grad','cr1_history_bias_grad','cr2_history_bias_grad']].agg(['mean','max']))


## 13. Same-weights destructive perturbation test

This is the strongest causal test. We keep the trained H3 weights fixed and change only the evidence:

- `correct_history`
- `zero_all_history`
- `zero_node_history`
- `zero_projected_support`
- `zero_new_edge_features`
- `shuffled_history`

If correct history is materially better than zero/shuffled history, the model is genuinely using identity-specific historical evidence.

In [ ]:
def clone_batch_cpu(src):return {k:(v.clone() if torch.is_tensor(v) else v) for k,v in src.items()}
def perturb(src,mode):
    x=clone_batch_cpu(src)
    if mode=='correct_history':return x
    if mode=='zero_all_history':
        x['node_history_valid'].zero_();x['history_support_valid'].zero_();x['hypothesis_edge_attr'][:,8:22].zero_();x['best_current_component_id'].fill_(-1);x['best_component_overlap'].zero_();x['second_best_component_overlap'].zero_();return x
    if mode=='zero_node_history':x['node_history_valid'].zero_();return x
    if mode=='zero_projected_support':x['history_support_valid'].zero_();return x
    if mode=='zero_new_edge_features':x['hypothesis_edge_attr'][:,8:22].zero_();return x
    if mode=='shuffled_history':
        g=torch.Generator().manual_seed(SEED+99);ids=torch.nonzero(x['node_history_valid']).flatten()
        if len(ids)>1:
            p=ids[torch.randperm(len(ids),generator=g)];tmp=x['node_instance_grid'][ids].clone();x['node_instance_grid'][ids]=tmp[torch.randperm(len(ids),generator=g)]
        m=len(x['history_support'])
        if m>1:
            p=torch.randperm(m,generator=g)
            for k in ['history_support','history_support_valid','history_support_dt','history_support_center_um','history_support_extent_um','best_current_component_id','best_component_overlap','second_best_component_overlap']:x[k]=x[k][p].clone()
        return x
    raise KeyError(mode)

@torch.no_grad()
def perturb_eval(mode):
    pb=move_batch_to_device(perturb(batch,mode),device);model.eval();criterion=trainer.criterion
    with torch.autocast('cuda',dtype=torch.float16):o=model_forward_from_batch(model,pb);losses=criterion(o,pb['targets'])
    matches=get_matches(o,criterion);qs=seed_queries(o,9);masks={q:native_query_mask(o,q) for q in qs};pairs=[sparse_dice(masks[a],masks[c]) for i,a in enumerate(qs) for c in qs[i+1:]]
    mmap={int(q):int(t) for q,t in zip(matches[0].pred_indices.detach().cpu(),matches[0].target_indices.detach().cpu())};sr=source_row[9];comp=torch.nonzero(source_overlap[sr]>0).flatten().tolist();assigned=[]
    for q in qs:
        t=mmap.get(q,-1)
        if t in comp:assigned.append(q_gt_dice(masks[q],t))
    row={'mode':mode,'loss':float(losses['loss']),'coarse_soft_dice':1-float(losses['dice_coarse']),'native_soft_dice':1-float(losses['dice_hi']),
         'source9_pair_mean':float(np.mean(pairs)),'source9_pair_median':float(np.median(pairs)),'source9_assigned_mean_dice':float(np.mean(assigned)) if assigned else np.nan,
         'matched_count':len(matches[0].pred_indices)}
    del pb,o,losses,matches,masks;gc.collect();torch.cuda.empty_cache();return row

modes=['correct_history','zero_all_history','zero_node_history','zero_projected_support','zero_new_edge_features','shuffled_history']
perturb_df=pd.DataFrame([perturb_eval(m) for m in modes]);display(perturb_df);perturb_df.to_csv(RUN_DIR/'same_weights_history_perturbations.csv',index=False)


## 14. Source-9 representation trajectory

High embedding cosine + low center separation + high mask overlap means split siblings are still homogenized. If history is useful, those quantities should move in the opposite direction.

In [ ]:
display(collapse_df[collapse_df.snapshot.isin(['post_spatial_before_history_training','after_temporal_dense','after_query_bootstrap','after_native_bootstrap','after_joint'])])


## 15. Automatic summary

In [ ]:
summary={'preflight':preflight,'source9_projected_tracklets':len(s9_history),'final_snapshot':snapshot_df.iloc[-1].to_dict(),'perturbations':perturb_df.to_dict(orient='records')}
if nb12_snap_path.exists():summary['notebook12_final']=pd.read_csv(nb12_snap_path).iloc[-1].to_dict()
with open(RUN_DIR/'summary.json','w',encoding='utf-8') as f:json.dump(summary,f,indent=2,default=float)
print(json.dumps(summary,indent=2,default=float))


# How to interpret the run

### Strong evidence the new history architecture is useful

- several distinct tracklets project onto source 9;
- history gradients are nonzero;
- Notebook 13 beats Notebook 12 on source-9 separation and/or native Dice;
- **correct history beats zero/shuffled history with identical trained weights**;
- source-9 center separation increases and sibling native-mask Dice falls.

### History is reaching the model but split symmetry still fails

If correct history is better than zero/shuffled history, but source-9 sibling overlap remains near the Notebook-12 value (~0.9), then the history pathway is useful but does not solve the query symmetry problem. The likely next architectural change is to give split queries distinct physical/history anchors.

### History is ignored

If valid history is abundant but correct/zero/shuffled perturbations are almost identical and history gates/biases remain near zero, the architecture is not exploiting the new pathway.

### Upstream evidence shortage

If very few distinct tracklets project onto source 9, the limitation is evidence availability rather than the neural module itself.

## Send back

Please send the output of:

1. the real-history preflight table;
2. source-9 projected-tracklet table;
3. final Notebook-12 vs Notebook-13 comparison;
4. assignment-switch table;
5. history-gradient table;
6. same-weights perturbation table;
7. source-9 representation trajectory.

Those are sufficient to decide the next model change.